# CatBoost for Malware Detection

This notebook implements CatBoost (Categorical Boosting) for binary classification of malware vs goodware.

CatBoost is a gradient boosting library that handles categorical features automatically and is robust to overfitting.

**Note:** CatBoost may not be compatible with Python 3.14+. If you encounter installation issues, use Python 3.13 or lower.

## Steps:
1. Load and explore the dataset
2. Preprocess the data
3. Split into train/test sets (80/20)
4. Train CatBoost model
5. Evaluate using 10-fold cross-validation
6. Test on hold-out test set
7. Analyze feature importance
8. Save the model

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

# Try to import CatBoost
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
    print("CatBoost imported successfully!")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("WARNING: CatBoost not available. Please install it with: pip install catboost")
    print("Note: CatBoost requires Python 3.13 or lower.")

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/brazilian-malware-dataset/brazilian-malware.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check class distribution
print("Class distribution:")
print(df['Label'].value_counts())
print(f"\nClass balance: {df['Label'].value_counts(normalize=True)}")

## 2. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Label', axis=1)

# Keep only numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns
X = X[numeric_cols]

y = df['Label']

# Handle missing values
if X.isnull().sum().sum() > 0:
    print("Filling missing values with 0...")
    X = X.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 3. Train/Test Split (80/20)

In [ ]:
# Split into train and test sets (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

## 4. Feature Scaling

Note: CatBoost doesn't require scaling, but we'll do it for consistency.

In [ ]:
# Initialize and fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")

## 5. Train CatBoost Model

In [ ]:
if not CATBOOST_AVAILABLE:
    print("CatBoost is not available. Please install it to continue.")
else:
    # Initialize CatBoost model with optimized hyperparameters
    cb_model = CatBoostClassifier(
        iterations=100,          # Number of boosting rounds
        depth=6,                 # Maximum tree depth
        learning_rate=0.1,       # Step size shrinkage
        random_seed=RANDOM_STATE,
        verbose=False,           # Suppress training output
        thread_count=-1          # Use all CPU cores
    )
    
    # Train the model
    print("Training CatBoost model...")
    cb_model.fit(X_train_scaled, y_train)
    print("Model training complete!")

## 6. Cross-Validation Evaluation (10-Fold)

In [ ]:
if CATBOOST_AVAILABLE:
    # Perform 10-fold stratified cross-validation
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    
    print("Performing 10-fold cross-validation...")
    print("This may take a few minutes...")
    cv_results = cross_validate(
        cb_model, 
        X_train_scaled, 
        y_train,
        cv=cv,
        scoring=['roc_auc', 'accuracy'],
        n_jobs=-1,
        return_train_score=False
    )
    
    # Display results
    print("\n=== Cross-Validation Results ===")
    print(f"AUC: {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")
    print(f"Accuracy: {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")

## 7. Final Evaluation on Hold-Out Test Set

In [ ]:
if CATBOOST_AVAILABLE:
    # Make predictions on test set
    y_pred = cb_model.predict(X_test_scaled)
    y_pred_proba = cb_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    test_accuracy = accuracy_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_pred_proba)
    
    print("\n=== Test Set Results ===")
    print(f"Accuracy: {test_accuracy:.4f}")
    print(f"AUC: {test_auc:.4f}")

In [ ]:
if CATBOOST_AVAILABLE:
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)
    
    # Visualize confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Goodware', 'Malware'],
                yticklabels=['Goodware', 'Malware'])
    plt.title('Confusion Matrix - CatBoost')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

In [ ]:
if CATBOOST_AVAILABLE:
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Goodware', 'Malware']))

## 8. Feature Importance Analysis

In [ ]:
if CATBOOST_AVAILABLE:
    # Get feature importances
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': cb_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))
    
    # Visualize top features
    plt.figure(figsize=(10, 6))
    top_features = feature_importance.head(15)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance')
    plt.title('Top 15 Feature Importances - CatBoost')
    plt.tight_layout()
    plt.show()

## 9. Model Parameters

In [ ]:
if CATBOOST_AVAILABLE:
    # Display model parameters
    print("Model Hyperparameters:")
    print(f"Number of iterations: {cb_model.get_param('iterations')}")
    print(f"Depth: {cb_model.get_param('depth')}")
    print(f"Learning rate: {cb_model.get_param('learning_rate')}")
    print(f"\nNumber of trees: {cb_model.tree_count_}")

## 10. Save Model

In [ ]:
if CATBOOST_AVAILABLE:
    # Create models directory if it doesn't exist
    import os
    os.makedirs('../models', exist_ok=True)
    
    # Save the model
    with open('../models/catboost.pkl', 'wb') as f:
        pickle.dump(cb_model, f)
    print("Model saved to ../models/catboost.pkl")
    
    # CatBoost also has its own save format
    cb_model.save_model('../models/catboost.cbm')
    print("Model also saved in CatBoost format to ../models/catboost.cbm")

## Summary

This notebook demonstrated:
1. ✅ Loading and exploring the malware dataset
2. ✅ Preprocessing with proper train/test split
3. ✅ Training a CatBoost classifier
4. ✅ 10-fold stratified cross-validation
5. ✅ Evaluation on hold-out test set
6. ✅ Feature importance analysis
7. ✅ Model serialization for production use

**Key Results:**
- Cross-Validation AUC: [See results above]
- Test Set AUC: [See results above]
- Test Set Accuracy: [See results above]

**Advantages of CatBoost:**
- **Handles categorical features** automatically without preprocessing
- **Robust to overfitting** - uses ordered boosting
- **No need for extensive hyperparameter tuning** - good defaults
- **Fast prediction** - optimized for inference
- **Handles missing values** natively
- **GPU support** for faster training

**Key Features:**
- Ordered boosting to reduce overfitting
- Symmetric trees for faster prediction
- Built-in handling of categorical features
- Multiple evaluation metrics
- Visualization tools included

**CatBoost vs XGBoost vs LightGBM:**
- **CatBoost**: Best for categorical features, most robust to overfitting
- **XGBoost**: Most mature, widely used, good all-around performance
- **LightGBM**: Fastest training, best for very large datasets

**When to Use CatBoost:**
- Datasets with categorical features
- Need for robust model with minimal tuning
- Overfitting is a concern
- Fast inference is important

**Installation Note:**
If CatBoost is not available, it may be due to Python version incompatibility. CatBoost requires Python 3.13 or lower. Install with:
```bash
pip install catboost
```